# 🎙️ YouTube Hindi / English Transcriber (free, Google Colab)

Paste YouTube links → get transcripts (TXT, timestamps, SRT). Runs on Google's free GPU with **Whisper large-v3**, good for Hindi, English and Hinglish, and handles hours of audio.

**How to use (works on iPad Safari):**
1. Tap ▶ on **Step 1** (once per session, ~1–2 min). If asked *"Run anyway?"* tap **Run anyway**.
2. Paste your link(s) in **Step 2**, pick the language, tap ▶.
3. Transcripts appear below and download as a ZIP (or go to Google Drive if you tick that box).

Tips: Keep this tab open while it runs. If YouTube blocks the download, use **Runtime → Disconnect and delete runtime**, then run Step 1 again (you get a new machine). Web app version: https://youtube-transcript-live.streamlit.app

In [ ]:
#@title **Step 1 — Setup** (run once per session)
import subprocess, sys, os, shutil
print("Installing tools…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp[default]", "deno", "faster-whisper"], check=True)
if os.path.isdir("/content/youtube-transcriber"):
    subprocess.run(["git", "-C", "/content/youtube-transcriber", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/agshivam/youtube-transcriber", "/content/youtube-transcriber"], check=True)
sys.path.insert(0, "/content/youtube-transcriber")
import importlib, transcriber
importlib.reload(transcriber)
gpu = shutil.which("nvidia-smi") and subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none (slower). For speed: Runtime → Change runtime type → T4 GPU, then run Step 1 again.")
print("FFmpeg:", "ok" if shutil.which("ffmpeg") else "missing")
print("✅ Setup done. Now run Step 2.")

In [ ]:
#@title **Step 2 — Paste YouTube link(s) and run**
links = "https://youtu.be/Jt-cZQjUdaY" #@param {type:"string"}
language = "Hindi" #@param ["Hindi", "English", "Auto (Hindi / English / Hinglish)"]
model_size = "large-v3" #@param ["large-v3", "large-v3-turbo", "medium", "small"]
use_youtube_captions_if_available = False #@param {type:"boolean"}
save_to_google_drive = False #@param {type:"boolean"}
#@markdown Separate several links with spaces, commas or new lines. Leave *use YouTube captions* off for the best Hindi quality (YouTube's auto-captions are often poor).

import re, sys, time, tempfile, zipfile, shutil
from pathlib import Path
sys.path.insert(0, "/content/youtube-transcriber")
import transcriber as T

LANG = {"Hindi": "hi", "English": "en"}.get(language)
OUT = Path("/content/transcripts"); OUT.mkdir(parents=True, exist_ok=True)

def get_model():
    global _MODEL, _MODEL_NAME, _RUNTIME
    if globals().get("_MODEL_NAME") != model_size:
        print(f"Loading Whisper {model_size} (first time downloads ~1.5 GB)…")
        _MODEL, _RUNTIME = T.load_whisper_model(model_size)
        _MODEL_NAME = model_size
        print("Model ready on", _RUNTIME)
    return _MODEL

def whisper_transcribe(path, title):
    model = get_model()
    gpu = "GPU" in _RUNTIME
    segments, info = model.transcribe(str(path), language=LANG, beam_size=5, vad_filter=True,
                                      vad_parameters={"min_silence_duration_ms": 500},
                                      condition_on_previous_text=True)
    total = max(float(getattr(info, "duration", 0) or 0), 1.0)
    done, last = [], 0.0
    started = time.time()
    for seg in segments:
        done.append(seg)
        pct = min(seg.end / total * 100, 100)
        if pct - last >= 5 or pct >= 100:
            print(f"   {pct:5.1f}%  ({T.format_short_timestamp(seg.end)} of {T.format_short_timestamp(total)}, {time.time()-started:.0f}s)")
            last = pct
    return T.build_outputs(title, done, getattr(info, "language", LANG), source=f"Whisper {model_size} on {_RUNTIME}")

def transcribe_link(url):
    notes = []
    if use_youtube_captions_if_available:
        result = T.fetch_youtube_captions(url, LANG, notes)
        for n in notes: print("   " + n)
        if result is not None and (LANG is None or result.detected_language == LANG):
            return result
    with tempfile.TemporaryDirectory() as tmp:
        print("   Downloading audio…")
        audio, title = T.download_youtube_audio(url, tmp)
        print(f"   Got audio: {title}")
        return whisper_transcribe(audio, title)

urls = [u for u in re.split(r"[\s,]+", links.strip()) if u]
if not urls:
    raise SystemExit("Paste at least one YouTube link above.")

saved, failed = [], []
for i, url in enumerate(urls, 1):
    print(f"\n[{i}/{len(urls)}] {url}")
    try:
        r = transcribe_link(url)
    except T.TranscriptionError as exc:
        msg = str(exc)
        if T._is_blocked(exc):
            msg += "\n   → YouTube blocked this machine. Runtime → Disconnect and delete runtime, then run Step 1 and Step 2 again."
        print("   ❌", msg); failed.append(url); continue
    for name, text in ((r.txt_filename, r.clean_text), (r.timestamped_filename, r.timestamped_text), (r.srt_filename, r.srt_text)):
        (OUT / name).write_text(text, encoding="utf-8")
        saved.append(OUT / name)
    print(f"   ✅ {r.title} — {r.source}, language: {r.detected_language}")
    print("   ---- transcript ----")
    print(r.clean_text)
    print("   --------------------")

if saved:
    if save_to_google_drive:
        from google.colab import drive
        drive.mount("/content/drive")
        dest = Path("/content/drive/MyDrive/YouTube Transcripts"); dest.mkdir(parents=True, exist_ok=True)
        for f in saved: shutil.copy(f, dest / f.name)
        print(f"\n📁 Saved {len(saved)} files to Google Drive → MyDrive/YouTube Transcripts")
    zip_path = Path("/content/transcripts.zip")
    with zipfile.ZipFile(zip_path, "w") as z:
        for f in saved: z.write(f, f.name)
    try:
        from google.colab import files
        files.download(str(zip_path))
        print("⬇️ Downloading transcripts.zip (also in the 📁 Files panel on the left → transcripts).")
    except Exception:
        print("Files are in /content/transcripts")
print(f"\nDone: {len(urls) - len(failed)} succeeded, {len(failed)} failed.")